In [68]:
import gensim
from gensim import corpora
from gensim.utils import simple_preprocess
from gensim.models import LdaModel

In [ ]:
# Загружаем стоп-слова
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

In [70]:
stop_words = stopwords.words('english')

In [3]:
documents = [
    "The economy is growing rapidly and people are spending more money.",
    "Sports are a great way to stay fit and have fun with friends.",
    "Technology is advancing quickly, and new gadgets are being released every year.",
    "The political situation is unstable in many countries around the world.",
    "Health and fitness are important for a long and happy life."
]

__________________________________________________________

In [71]:
# Преобразовываем каждое предложение в список слов
def preprocess_text(sentences):
    for sentence in sentences:
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))
        # Ключевое слово yield используется для создания генератора. 
        # Вместо того чтобы возвращать сразу весь результат, 
        # генератор возвращает обработанные предложения по одному, 
        # позволяя экономить память при обработке больших объемов данных.

In [ ]:
processed_docs = list(preprocess_text(documents))
processed_docs

In [77]:
# Очищаем от стоп сллов
def remove_stopwords(texts):
    return [[word for word in doc if word not in stop_words] for doc in texts]

In [ ]:
data_words_nostops = remove_stopwords(processed_docs)
data_words_nostops

In [79]:
# Создаем словарь и корпус
dictionary = corpora.Dictionary(data_words_nostops)
corpus = [dictionary.doc2bow(doc) for doc in data_words_nostops]

In [80]:
# Обучаем LDA-модель
num_topics = 3
lda_model = LdaModel(corpus, num_topics=num_topics, id2word=dictionary, passes=15, random_state=42)

In [81]:
# Вывод доминирующих тем для каждого документа
def get_dominant(lda_model, corpus):
    dominant_topics = []
    for doc in corpus:
        topic_probs = lda_model.get_document_topics(doc)
        dominant_topic = max(topic_probs, key=lambda x: x[1])[0]
        dominant_topics.append(dominant_topic)
    return dominant_topics

Пояснение как работает функция get_dominant  
1. Входные параметры:  
- <font color='lightgreen'>lda_model</font> - обученная модель LDA (LdaModel), которая может генерировать вероятности тем для каждого документа.  
- <font color='lightgreen'>corpus</font> - корпус документов, представленный в виде мешка слов (bag-of-words), где каждый документ представлен как список пар (идентификатор слова, частота).  

2. <font color='lightgreen'>dominant_topics</font> - (изачально пустой) будет хранить индексы доминирующих тем для каждого документа.  
3. <font color='lightgreen'>for doc in corpus</font> - для каждого документа в корпусе вычисляются вероятности принадлежности документа к каждой теме с помощью метода get_document_topics:  
   - <font color='lightgreen'>topic_probs = lda_model.get_document_topics(doc)</font>  
topic_probs возвращает список пар (topic_id, probability), где topic_id — индекс темы, а probability — вероятность того, что документ относится к данной теме.  
    - <font color='lightgreen'>dominant_topic = max(topic_probs, key=lambda x: x[1])[0]</font>  
Функция max используется для поиска темы с максимальной вероятностью (второй элемент пары (topic_id, probability)).
<font color='lightgreen'>key=lambda x: x[1]</font> указывает, что максимальное значение определяется по вероятностям (второму элементу пары).
[0] извлекает идентификатор темы (первый элемент пары).
    - <font color='lightgreen'>dominant_topics.append(dominant_topic)</font>  
Индекс доминирующей темы для текущего документа добавляется в список dominant_topics.

3. Пример работы:
Если topic_probs для документа равно [(0, 0.2), (1, 0.6), (2, 0.2)], то:  
(темы 0, 1, 2;  вероятности  относения документа к данной теме - 0.2, 0.6, 0.2)
Максимальная вероятность — 0.6.
Соответствующий topic_id —№ 1.
Для этого документа доминирующая тема — 1.

In [82]:
dominant_topics = get_dominant(lda_model, corpus)

In [85]:
dominant_topics

[2, 1, 1, 2, 2]

In [83]:
# Вывод результатов
print("Dominant topics in each document:")
for i, topic in enumerate(dominant_topics):
    print(f"Document {i + 1}: Topic {topic}")

Dominant topics in each document:
Document 1: Topic 2
Document 2: Topic 1
Document 3: Topic 1
Document 4: Topic 2
Document 5: Topic 2


In [84]:
# Вывод топ-слов для каждой темы
print("\nTop words for each topic:")
for i, topic in enumerate(lda_model.show_topics(num_topics=num_topics, num_words=5, formatted=True)):
    print(f"Topic {i}: {topic[1]}")


Top words for each topic:
Topic 0: 0.029*"health" + 0.029*"happy" + 0.029*"economy" + 0.029*"rapidly" + 0.029*"long"
Topic 1: 0.051*"released" + 0.051*"technology" + 0.051*"gadgets" + 0.051*"every" + 0.051*"quickly"
Topic 2: 0.044*"countries" + 0.044*"unstable" + 0.044*"around" + 0.044*"many" + 0.044*"situation"


In [86]:
# Вывод нескольких доминирующих тем для каждого документа
def get_top_topics(lda_model, corpus, top_n=2):
    top_topics_list = []
    for doc in corpus:
        topic_probs = lda_model.get_document_topics(doc)
        top_topics = sorted(topic_probs, key=lambda x: x[1], reverse=True)[:top_n]
        top_topics_list.append([topic[0] for topic in top_topics])
    return top_topics_list

In [87]:
top_topics_list = get_top_topics(lda_model, corpus, top_n=2)

In [88]:
top_topics_list

[[2, 0], [1, 0], [1, 0], [2, 0], [2, 0]]

In [89]:
# Вывод результатов
print("Top topics in each document:")
for i, topics in enumerate(top_topics_list):
    print(f"Document {i + 1}: Topics {topics}")

Top topics in each document:
Document 1: Topics [2, 0]
Document 2: Topics [1, 0]
Document 3: Topics [1, 0]
Document 4: Topics [2, 0]
Document 5: Topics [2, 0]


In [90]:
# Вывод топ-слов для каждой темы
print("\nTop words for each topic:")
for i, topic in enumerate(lda_model.show_topics(num_topics=num_topics, num_words=5, formatted=True)):
    print(f"Topic {i}: {topic[1]}")


Top words for each topic:
Topic 0: 0.029*"health" + 0.029*"happy" + 0.029*"economy" + 0.029*"rapidly" + 0.029*"long"
Topic 1: 0.051*"released" + 0.051*"technology" + 0.051*"gadgets" + 0.051*"every" + 0.051*"quickly"
Topic 2: 0.044*"countries" + 0.044*"unstable" + 0.044*"around" + 0.044*"many" + 0.044*"situation"
